# Introduction to STARSHIPS Reduction

**STARSHIPS**  
*Spectral Transmission and Radiation Search for High Resolution Planet Signal*

STARSHIPS is a pipeline designed to remove the stellar spectrum signature and residuals from high-resolution time-series data to extract the planetary signal. It generates synthetic sequences and prepares the data for cross-correlation, working order-by-order.

The reduction process consists of several main steps:
1. **Bad Pixel Correction and Masking:** Identify and mask bad pixels in the data.
2. **Spectral Alignment:** Shift each spectrum to align the stellar signal by correcting for barycentric Earth radial velocity (BERV).
3. **Telluric Line Masking:** Identify and mask regions affected by deep telluric absorption.
4. **Reference Spectrum Removal:** Build a master reference spectrum from the aligned spectra and subtract it.
5. **PCA Residual Removal:** Apply principal component analysis (PCA) to remove quasi-static vertical residuals. Typically, multiple numbers of principal components (nPC) are tested.

---

## Working with SPIRou or NIRPS Data

- Data format is typically (number of exposures, number of orders, number of pixels), with >4000 pixels per order.
- Initial calibrations are performed by the **APERO** pipeline.

**APERO** performs:
- Dark correction, bad pixel correction, background subtraction
- Detector nonlinearity correction, order localization, geometric correction
- Flat field correction, blaze correction, hot pixel and cosmic ray removal
- Telluric correction
- Wavelength calibration using the Fabry-Perot etalon method

Reference: [APERO Pipeline Paper](https://iopscience.iop.org/article/10.1088/1538-3873/ac9e74)

APERO produces three main data products:
- **E2DS**: Original extracted spectra
- **TCORR**: Spectra corrected for tellurics
- **RCONN**: Synthetic telluric spectrum

The wavelength solution is handled separately. STARSHIPS will either use the available solution or reconstruct it from the file headers using the `planet_obs.py` functions.

---

## Data Organization and Preprocessing

- APERO outputs are organized by target, not by observation date.
- Observations must be split into individual nights/visits using the "split nights".
- This step sorts by Barycentric Julian Date (BJD) and reduction type, identifying observation gaps.

The sorted observations are saved into files named like:

These files are required for input into STARSHIPS.

Each notebook typically runs one visit reduction at a time.

---

## Adapting STARSHIPS for Other Instruments

STARSHIPS is designed to be flexible:
- Supported instruments currently include SPIRou, NIRPS (APERO and Geneva/ESPRESSO reductions), and IGRINS.
- Modifications to instrument headers can be made inside `planet_obs.py`.



<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Welcome to the workshop version of this notebook! This notebook will NOT fully run through. We have placed highlighted questions to help guide you with what needs to be fixed/changed. If needed feel free to take a peek at the answer key if you are stuck!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  all the answers/ hints will be shown in these drop down menus!
</details>


In [ ]:
# We are NOT running petitRADTRANS in this tutorial. However, starships needs it to run. 
# We are tricking the container here to believe we have petitRADTRANS when in fact all the functions are empty


import sys
import types

# Top-level dummy package
petitRADTRANS = types.ModuleType("petitRADTRANS")

# Submodules as separate mock modules/namespaces
petitRADTRANS.Radtrans = object  # or define a dummy class

petitRADTRANS.nat_cst = types.SimpleNamespace()
petitRADTRANS.physics = types.SimpleNamespace(
    guillot_global=lambda *a, **k: None,
    guillot_modif=lambda *a, **k: None
)
petitRADTRANS._read_opacities = types.SimpleNamespace()
petitRADTRANS.fort_input = types.SimpleNamespace()
petitRADTRANS.fort_rebin = types.SimpleNamespace()
petitRADTRANS.pyth_input = types.SimpleNamespace()

poor_mans_nonequ_chem = types.ModuleType("poor_mans_nonequ_chem")
poor_mans_nonequ_chem.interpol_abundances = lambda *a, **k: None

# Register everything in sys.modules
sys.modules["petitRADTRANS"] = petitRADTRANS
sys.modules["petitRADTRANS.radtrans"] = petitRADTRANS
sys.modules["petitRADTRANS._read_opacities"] = petitRADTRANS._read_opacities
sys.modules["petitRADTRANS.fort_input"] = petitRADTRANS.fort_input
sys.modules["petitRADTRANS.fort_rebin"] = petitRADTRANS.fort_rebin
sys.modules["petitRADTRANS.pyth_input"] = petitRADTRANS.pyth_input
sys.modules["petitRADTRANS.nat_cst"] = petitRADTRANS.nat_cst
sys.modules["petitRADTRANS.physics"] = petitRADTRANS.physics
sys.modules["petitRADTRANS.poor_mans_nonequ_chem"] = poor_mans_nonequ_chem

In [ ]:
# === System and Path Utilities ===
import os
from sys import path
from pathlib import Path

# === Suppress Warnings ===
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

# === Numerical and Scientific Libraries ===
import numpy as np
import astropy.units as u
import astropy.constants as const

# === Plotting ===
import matplotlib.pyplot as plt

# === STARSHIPS Core Imports ===
import starships.homemade as hm
import starships.planet_obs as pl_obs
import starships.plotting_fcts as pf
from starships import spectrum as spectrum
from starships.planet_obs import Observations
from starships.mask_tools import interp1d_masked

# Disable verbose interpolation output
interp1d_masked.iprint = False


In [ ]:
# Confirm the path to this notebook within the container
print(os.getcwd())

# Let's set up the planet name and reduction name. This way, we can set up the directory in which projects will be stored

In [ ]:
# Planet name as used in exofile
pl_name = 'WASP-127 b'

# Reduction name
reduction = 'ExoSLAM_Workshop'

# Output reductions in dedicated directory
pl_name_fname = ''.join(pl_name.split())  # Remove spaces from the planet name
out_dir = Path(f"{pl_name_fname}_{reduction}")  # Convert to a Path object

# Make sure the  directory exists
out_dir.mkdir(parents=True, exist_ok=True) 
print(out_dir)

# Load in the filenames lists that have been previously created in split nights. 
For the purpose of this workshop, we have done this for you

In [ ]:
# Make sure the  directory exists
obs_dir = '/home/jovyan/WASP-127data'

# Where to save figures?
path_fig = Path(f"{pl_name_fname}_{reduction}")

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
What are the file list names? Fill them in for <code>e2d</code> and <code>tcorr</code>. What do you think these file lists are and what does each file contain?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  The files are listed within the container in the folder "WASP-127data. The names are 'list_e2ds_2023-03-10-00' and 'list_tcorr_2023-03-10-00'. one list has the non telluric corrected data and the other the telluric corrected data.
</details>


In [ ]:
list_filenames = {'list_e2ds': 'list_"insert file name here"',
                  'list_tcorr':'list_'"insert file name here"'}
print(list_filenames)

# Let's load in the Planet Parameters

At this stage, we need to define the planet parameters.
We can either load them from an existing Exofile or manually reassign them as shown below.
The function used to load the data is Observations, and its inputs are listed below.

The function needed here is "Observations" and the inputs of the function can be seen in the cell below.

In [ ]:
Observations?

<span style="font-size:20px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
These are the parameters for the wrong planet! Can you fix them? Is the <code>pl_kwargs</code> list correct, or is it missing any of the defined parameters? Is the instrument name correct?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Check the answer key for the correct parameters! What needs to be changed it M_star, R_star, M_pl, R_pl and period. The missing parameters in the <code>pl_kwargs</code>  are M_pl, and ap. Yes, the instrument name is correct.
  
</details>


In [ ]:

# === Planetary and Stellar Parameters that can be changed===
M_star = 1.950 * u.M_sun           # Star mass
R_star = 10.333 * u.R_sun           # Star radius
Teff = 5842 * u.K                  # Star effective temperature

M_pl = 15.165 * u.M_jup             # Planet mass
R_pl = 100.311 * u.R_jup             # Planet radius
Tp = 1400 * u.K                    # Planet temperature

period = 14.178062 * u.day          # Orbital period
trandur = 0.181 * u.day            # Transit duration
ap = 0.04840 * u.au                # Semi-major axis

incl = 87.85 * u.deg               # Orbital inclination
excent = 0.0                       # Eccentricity
w = (-90 * u.deg).to(u.rad)        # Argument of periastron (converted to radians)

# Mid transit time does not need to be changed for this data set, but this is how you could do it
# mid_tr = 2460158.66104             # Mid-transit time (BJD_TDB)
# t_peri = 2460158.66104             # Time of periastron passage (same as mid-transit for circular orbit)




# === Create Observation Object from above parameters ===
obs = Observations(
    name=pl_name,
    instrument='NIRPS-APERO',
    pl_kwargs={
        # 'mid_tr': mid_tr,
        # 't_peri': t_peri,
        'excent': excent,
        'w': w,
        'M_star': M_star,
        'R_star': R_star,
        'Teff': Teff,
    
        'R_pl': R_pl,
        'Tp': Tp,
        'period': period,
        'trandur': trandur,

        'incl': incl
    }
)



In [ ]:
# Let's re-check those parameters so that we are certain they did indeed change

p = obs.planet

print(p.M_star)
print(p.R_star)
print(p.Teff)

print(p.M_pl)
print(p.R_pl)
print(p.Tp)

print(p.period)
print(p.trandur)
print(p.ap)

print(p.incl)
print(p.excent)
print(p.w)

print(p.mid_tr)
print(p.t_peri)


# Next we fetch the observations. We can do this with:  
                    
                    
Note: The CADC will mean that this data is being pulled from the Archive

In [ ]:
obs.fetch_data?

In [ ]:
obs.fetch_data(obs_dir, CADC=True, **list_filenames)


# Do a quick sanity check to make sure all the data is as expected. We can plot it all below

In [ ]:
# Extract wave, blaze, and flux data from obs.__dict__
wave = obs.__dict__.get('wave', None)
blaze = obs.__dict__.get('blaze', None)
flux = obs.__dict__.get('count', None)
tellu= obs.__dict__.get('tellu', None)
uncorr= obs.__dict__.get('uncorr', None)


print("shape wave", np.shape(wave))
print("shape blaze",np.shape(blaze))
print("shape, flux", np.shape(flux))

In [ ]:
# Select one exposure and one order (e.g., exposure 0 and order 0)
exposure_index = 1
order_index = 41

# Extract the data for the selected exposure and order
selected_wave = wave[exposure_index, order_index]
selected_flux = flux[exposure_index, order_index]
selected_blaze = blaze[exposure_index, order_index]
selected_telluric= tellu[exposure_index, order_index]
selected_uncorr= uncorr[exposure_index, order_index]


# Plot the flux and blaze
plt.figure(figsize=(10, 5))
plt.plot(selected_wave, selected_flux/np.nanmean(selected_flux), label='Flux', alpha=0.7)
plt.plot(selected_wave, selected_blaze/np.nanmean(selected_blaze), label='Blaze', alpha=0.7)
plt.plot(selected_wave, selected_telluric, label='tellurics', alpha=0.7)
plt.plot(selected_wave, selected_uncorr/np.nanmean(selected_uncorr), label='uncorr', alpha=0.7)


plt.xlabel('Wavelength')
plt.ylabel('Normalized Flux')
plt.title(f'Exposure {exposure_index}, Order {order_index}')
plt.legend()
plt.grid()
plt.show()

## Let's take a look at the reduction functions that will be used below

In [ ]:
pl_obs.generate_all_transits?


In [ ]:
pl_obs.save_single_sequences?

In [ ]:
pl_obs.save_sequences?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Fill in parameters here for the Mask_wings, kind_trans, and RVsys. These are the parameters most likely needed to be changed from planet to planet when running new reductions!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Mask_winds can be 0.9,0.95, or 0.97! It can be other numbers too but these are the most standard. kind_trans should ='transmission' since we are looking at a transit. Rv sys can be found on the gaia website! it should be -8.9 [km/s].
</details>


In [ ]:


visit_name = 'WASP127b_TR1_data_trs_' #name your visit, this will be used in the final names
mask_wings =    # Fraction of telluric wings to mask [typical inputs include 0.90, 0.95, 0.97]

# Radial velocity injection range
n_RV_inj = 31
corrRV0 = np.linspace(-30, 30, n_RV_inj)

# Limb darkening 
coeffs = [0.23081159, 0.2778549]  # Limb-darkening coefficients, can be changed depending on planet
ld_model = 'quadratic'           # Limb-darkening model, can be changed depending on planet

# Correct bad pixels [True, False]
cbp = True

# Type of observation [Transmission, Emission]
kind_trans = '  '      

# System RV [Can be found in Gaia] in km/s
RVsys = [ ] 

# if multiple transits are loaded, used to split transits (for one transit put [None] unless you want to split the data set) 
# this can also be used to remove expsoures, here you fill in the index for all the exposures you would like to use
transit_tags = [None]

iout_all = ['all'] # exposures considered as part of master out
polynome = [False] # fit polynomial to remove low-frequency structures 

#only used if doing multiple reductions at a time-> if not keep fixed
do_tr = [1]

# Transit generation kwargs
kwargs_gen_tr = {
    'coeffs': coeffs,
    'ld_model': ld_model,
    'do_tr': do_tr,
    'kind_trans': kind_trans,
    'polynome': polynome,
    'cbp': cbp
}

# Timeseries construction kwargs
kwargs_build_ts = {
    'clip_ratio': 6, # PARAMETERS IDK HOW TO EXPLAIN
    'clip_ts': 6, # PARAMETERS IDK HOW TO EXPLAIN
    'unberv_it': True #would you like to do the berv correction[True,False]

}

######################################################################################################################################
# Loop over number of principal components to remove

for n_pc in range(2, 3):
    # Additional parameters used for data reduction:
    telluric_mask_frac = 0.20        # Transmission fraction of deep tellurics to mask typical inputs [0.2,0.5]
    smoothing_width = 51             # Width of smoothing kernel (fixed)
    extra_param = 42                 # extra parameter (solution to the galaxy)
    gaussian_width = 5               # Gaussian kernel width for filtering (fixed)
    sigma_clip = 5.0                 # Sigma clipping threshold (fixed)

    # Combine all into param structure used by function
    params_all = [[
        telluric_mask_frac,    # [0]- Telluric fraction to mask
        mask_wings,            # [1] - Mask wings limit
        smoothing_width,       # [2] - Low-pass smoothing width
        extra_param,           # [3] - solution to the galaxy
        gaussian_width,        # [4] - Gaussian kernel width
        n_pc,                  # [5] - Number of principal components to remove
        sigma_clip, sigma_clip, sigma_clip, sigma_clip  # [6-9] - Clipping parameters at multiple steps set to value from above
    ]]

    # Generate transit sequences
    list_tr = pl_obs.generate_all_transits(
        obs, transit_tags, RVsys, params_all, iout_all,
        **kwargs_gen_tr, **kwargs_build_ts
    )

    # Save full transit sequence
    out_filename = f'sequence_{n_pc}-pc_mask_wings{int(mask_wings*100)}'
    pl_obs.save_single_sequences(out_filename, list_tr['1'], path=out_dir, filename_end=visit_name)

    # Save simplified version for retrieval use
    out_filename = f'retrieval_input_{n_pc}-pc_mask_wings{int(mask_wings*100)}'
    pl_obs.save_sequences(out_filename, list_tr, do_tr, path=out_dir)

# That's it! now let's see what we did

This can now be loaded in another notebook as well. Here we are analyzing the output of our reductions files that were created above and then stored.


In [ ]:
# This will plot the summary of mean SNR and airmass for the night
# In the plot red=out of transit, blue= in transit and green=ingress/egress
pl_obs.load_single_sequences?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
What is the file we just made? 
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>
Hint: leave off the data_trs_.npz the file name is sequence_2-pc_mask_wings90_data_trs_WASP127b_TR1_data_trs_.npz but here you can just fill in sequence_2-pc_mask_wings90_data_trs_WASP127b_TR1. The other two files we just created are just for running retrievals.
 </details>



In [ ]:
pl_obs.load_single_sequences('sequence_2-      ',
                             name=pl_name, planet=p, path=out_dir)


In [ ]:

pf.plot_steps?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Fill in the order for which you would like to see the reduction steps for!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Fill in the order= with any order number! ex: order=15
</details>

In [ ]:
Print('Number of orders in observations',obs.nord)


In [ ]:

order=
pf.plot_steps(list_tr['1'], order)
